Imports

In [5]:

%cd ..

from sklearn.model_selection import train_test_split
from src.preprocess import load_and_clean_data, add_new_features
from src.model import train_model, get_predictions_and_score
import src.plots as plots
import src.config as config

d:\CloudCredits Projects\Steel Industry Energy Consumption Analysis


 Cleaning and Preparing Data

In [6]:
# Load raw data and clean it
raw_df = load_and_clean_data()
df = add_new_features(raw_df)

print("Dataset cleaned and features added successfully!")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/Steel_industry_data.csv'

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
#Understanding Stats
print(df.describe())

In [ ]:
#Check for Missing Values:
print(df.isnull().sum())

In [ ]:

# Calculate correlation using only numeric columns
correlation_matrix = df.corr(numeric_only=True)

# Visualize with heatmap
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm')
plt.show()

In [ ]:
# Use 'dayfirst=True' to handle the 13/01 format correctly
df['date'] = pd.to_datetime(df['date'], dayfirst=True)

# Now you can safely extract the Hour and Day
df['Hour'] = df['date'].dt.hour
df['DayOfWeek'] = df['date'].dt.day_name()

print("Dates converted successfully!")

In [ ]:
# Check the first few rows to see the new columns
print(df[['date', 'Hour', 'DayOfWeek']].head())

In [ ]:
# Ensure the pivot table uses the newly created 'Hour' and 'DayOfWeek' columns
heatmap_data = df.pivot_table(index='DayOfWeek', columns='Hour', values='Usage_kWh', aggfunc='mean')

# Reorder days so they appear chronologically
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data.reindex(days_order)

# Plot
plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_data, cmap='YlGnBu', annot=False)

plt.title('Average Energy Consumption by Day and Hour')
plt.xlabel('Hour of the Day')
plt.ylabel('Day of the Week')
plt.show()

In [ ]:
# Convert to cyclical features to help the model understand that 23:00 is near 00:00
df['hour_sin'] = np.sin(2 * np.pi * df['Hour']/23)
df['hour_cos'] = np.cos(2 * np.pi * df['Hour']/23)

# Binary flag for weekend
df['is_weekend'] = df['DayOfWeek'].isin(['Saturday', 'Sunday']).astype(int)

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Define features (X) and target (y)
features = ['Hour', 'is_weekend', 'hour_sin', 'hour_cos'] # Add other sensor columns here
X = df[features]
y = df['Usage_kWh']

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the model
model = XGBRegressor(n_estimators=100, learning_rate=0.1)
model.fit(X_train, y_train)

# Predict
predictions = model.predict(X_test)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(y_test.values[:100], label='Actual Usage')
plt.plot(predictions[:100], label='Predicted Usage', linestyle='--')
plt.legend()
plt.title('Actual vs Predicted Energy Consumption')
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# 1. Date Conversion (Ensuring it handles your specific format)
df['date'] = pd.to_datetime(df['date'], dayfirst=True)

# 2. Encode Categorical Features
le = LabelEncoder()
df['WeekStatus_enc'] = le.fit_transform(df['WeekStatus'])
df['Day_of_week_enc'] = le.fit_transform(df['Day_of_week'])
df['Load_Type_enc'] = le.fit_transform(df['Load_Type'])

# 3. Feature Selection
# We exclude 'date' and the original string columns from the training
features = [
    'Lagging_Current_Reactive.Power_kVarh', 'Leading_Current_Reactive_Power_kVarh',
    'CO2(tCO2)', 'Lagging_Current_Power_Factor', 'Leading_Current_Power_Factor',
    'NSM', 'WeekStatus_enc', 'Day_of_week_enc', 'Load_Type_enc'
]

X = df[features]
y = df['Usage_kWh']

# 4. Split and Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = XGBRegressor(n_estimators=100, learning_rate=0.1)
model.fit(X_train, y_train)

# 5. Evaluate
predictions = model.predict(X_test)
print(f"R-squared Score: {r2_score(y_test, predictions):.4f}")

In [ ]:
# Plot feature importance
import xgboost as xgb
xgb.plot_importance(model)
plt.show()

In [ ]:
residuals = y_test - predictions
plt.scatter(predictions, residuals)
plt.axhline(0, color='red', linestyle='--')
plt.title('Residual Plot (Prediction Error)')
plt.show()

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, test_scores = learning_curve(model, X, y, cv=5)

plt.plot(train_sizes, train_scores.mean(axis=1), label='Training Score')
plt.plot(train_sizes, test_scores.mean(axis=1), label='Validation Score')
plt.title('Learning Curve (Checking for Overfitting)')
plt.legend()
plt.show()

In [ ]:
# Visualize top features
import pandas as pd
importance_df = pd.DataFrame({'Feature': features, 'Importance': model.feature_importances_})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.title('Key Drivers of Energy Consumption')
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

# 1. Initialize the models
models = {
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "LightGBM": LGBMRegressor()
}

# 2. Benchmark them against your test set
print(f"{'Model':<15} | {'R-squared Score':<15}")
print("-" * 35)

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    score = r2_score(y_test, preds)
    print(f"{name:<15} | {score:.4f}")